# Active Learning Experiment - Kaggle

**Metadata-Stratified Active Learning cho Object Detection**

Dataset: BDD100K | Model: YOLOv8

## 1. Cài đặt Dependencies

In [ ]:
!pip install ultralytics torch pandas pyyaml scipy scikit-learn --quiet

## 2. Clone Repository

In [ ]:
# Clone repo từ branch develop
!git clone -b develop https://github.com/nguyendinhtanloc-Org/metadata-stratified-active-learning.git

%cd /kaggle/working/metadata-stratified-active-learning

## 3. Import Modules

In [ ]:
# Add src to Python path - FIX: thêm parent directory
import sys

sys.path.insert(
    0,
    "/kaggle/working/metadata-stratified-active-learning"
)

# Import project modules
from active_learning.loop import ActiveLearningLoop
from sampling.baseline import baseline_sampling
from analysis.multi_seed_runner import (
    run_multi_seed_experiment,
    quick_test_config,
    compare_pipelines
)
from utils.config import DEFAULT_CONFIG

import pandas as pd
from pathlib import Path

print("Modules imported!")

## 4. Xác minh Dataset Paths

In [ ]:
# Kiểm tra đường dẫn dataset trên Kaggle
import os

DATASET_BASE = "/kaggle/input/datasets/solesensei/solesensei_bdd100k"

# Các đường dẫn cần kiểm tra
METADATA_PATH = f"{DATASET_BASE}/bdd100k_labels_release/bdd100k/labels/bdd100k_labels_images_train.json"
IMAGE_DIR = f"{DATASET_BASE}/bdd100k/bdd100k/images/100k/train"

# Kiểm tra tồn tại
print("Checking paths...")
print(f"Metadata exists: {os.path.exists(METADATA_PATH)}")
print(f"Image dir exists: {os.path.exists(IMAGE_DIR)}")

# Nếu không tồn tại, thử tìm đường dẫn khác
if not os.path.exists(METADATA_PATH):
    print("\nSearching for metadata file...")
    for root, dirs, files in os.walk(DATASET_BASE):
        for f in files:
            if f.endswith(".json") and "labels" in f:
                print(f"Found: {os.path.join(root, f)}")

if not os.path.exists(IMAGE_DIR):
    print("\nSearching for image directory...")
    for root, dirs, files in os.walk(DATASET_BASE):
        for d in dirs:
            if "train" in d.lower() or "images" in d.lower():
                print(f"Found: {os.path.join(root, d)}")

## 5. Cấu hình Experiment

In [ ]:
# Quick test config (batch nhỏ, ít epochs)
config = quick_test_config()

# Override cho Kaggle
config.update({
    "device": "0",  # GPU
    "imgsz": 640,
    "num_classes": 10,
})

print("Quick Test Config:")
for k, v in config.items():
    print(f"  {k}: {v}")

## 6. Chạy Experiment (Quick Test - 1 Seed)

In [ ]:
# Chạy stratified pipeline (quick test với 1 seed)
from active_learning.loop import ActiveLearningLoop

experiment_name = "kaggle_quick_test"
pipeline = "stratified"

print(f"Running {pipeline} experiment...")

al_loop = ActiveLearningLoop(
    config=config,
    experiment_name=experiment_name
)

results_stratified = al_loop.run(
    metadata_path=METADATA_PATH,
    image_dir=IMAGE_DIR
)

print("\nStratified Results:")
print(results_stratified)

## 7. Chạy Baseline Pipeline (Để So Sánh)

In [ ]:
# Chạy baseline pipeline (uncertainty-only, không stratified)
import active_learning.loop as loop_module

# Monkey-patch: thay stratified_sampling bằng baseline_sampling
loop_module.stratified_sampling = baseline_sampling

experiment_name_baseline = "kaggle_quick_test_baseline"

al_loop_baseline = ActiveLearningLoop(
    config=config,
    experiment_name=experiment_name_baseline
)

results_baseline = al_loop_baseline.run(
    metadata_path=METADATA_PATH,
    image_dir=IMAGE_DIR
)

print("\nBaseline Results:")
print(results_baseline)

## 8. So Sánh Hai Pipelines

In [ ]:
from analysis.statistics import format_statistical_report

# Thêm pipeline label
results_stratified["pipeline"] = "stratified"
results_baseline["pipeline"] = "baseline"

# So sánh
comparison = compare_pipelines(
    baseline_df=results_baseline,
    stratified_df=results_stratified
)

# In report
if "report" in comparison:
    print(comparison["report"])
else:
    print("\n=== Comparison ===")
    print(f"\nBaseline mAP50 mean: {results_baseline['map50'].mean():.4f}")
    print(f"Stratified mAP50 mean: {results_stratified['map50'].mean():.4f}")

## 9. Visualization

In [ ]:
import matplotlib.pyplot as plt

# Combine results
all_results = pd.concat([results_stratified, results_baseline], ignore_index=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: mAP Progress
ax1 = axes[0]
for pipeline in ["baseline", "stratified"]:
    data = all_results[all_results["pipeline"] == pipeline]
    ax1.plot(data["round"], data["map50"], marker="o", label=pipeline)

ax1.set_xlabel("Round")
ax1.set_ylabel("mAP@50")
ax1.set_title("Learning Curve Comparison")
ax1.legend()
ax1.grid(True)

# Plot 2: Batch Diversity (Entropy)
ax2 = axes[1]
for pipeline in ["baseline", "stratified"]:
    data = all_results[all_results["pipeline"] == pipeline]
    if "batch_entropy" in data.columns:
        ax2.plot(data["round"], data["batch_entropy"], marker="s", label=pipeline)

ax2.set_xlabel("Round")
ax2.set_ylabel("Batch Entropy")
ax2.set_title("Batch Diversity (Entropy)")
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig("results/comparison_plot.png", dpi=150)
plt.show()

print("Plot saved to results/comparison_plot.png")

## 10. Lưu Kết Quả

In [ ]:
# Tạo thư mục results
Path("results").mkdir(exist_ok=True)

# Lưu kết quả
results_stratified.to_csv("results/stratified_results.csv", index=False)
results_baseline.to_csv("results/baseline_results.csv", index=False)

print("Results saved:")
print("  - results/stratified_results.csv")
print("  - results/baseline_results.csv")
print("  - results/comparison_plot.png")

---

**Next Steps:**
1. Xác minh đường dẫn labels trên Kaggle
2. Chạy với nhiều seeds (3 seeds theo proposal)
3. Phân tích thống kê đầy đủ